In [ ]:
# Import libraries

import os
import json
import random

import numpy as np
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import monai
from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.metrics import DiceMetric
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    ConcatItemsd,
    NormalizeIntensityd,
    MapTransform,
    AsDiscrete,
    Compose,
)
from monai.inferers import sliding_window_inference

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"MONAI version: {monai.__version__}")
print(f"Using device: {device}")

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    "training_dir": os.path.join(BASE_DIR, "data", "brats2020", "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
    "configs": os.path.join(BASE_DIR, "configs"),
    "models": os.path.join(BASE_DIR, "models", "improved"),
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

In [ ]:
# Load the improved model and build the test dataset

class RemapLabeld(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            d[key][d[key] == 4] = 3
        return d


model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

model.load_state_dict(torch.load(os.path.join(PATHS["models"], "best_model.pth")))
model.eval()

print("Improved model loaded successfully")

# Load the test split
split_path = os.path.join(PATHS["configs"], "dataset_split.json")
with open(split_path, "r") as f:
    split_dict = json.load(f)

test_patients = split_dict["test"]
modalities = ["t1", "t1ce", "t2", "flair"]
patch_size = (96, 96, 96)

def build_data_dicts(patient_list, training_dir):
    data_dicts = []
    for pid in patient_list:
        patient_dir = os.path.join(training_dir, pid)
        entry = {mod: os.path.join(patient_dir, f"{pid}_{mod}.nii") for mod in modalities}
        entry["label"] = os.path.join(patient_dir, f"{pid}_seg.nii")
        entry["patient_id"] = pid
        data_dicts.append(entry)
    return data_dicts

test_dicts = build_data_dicts(test_patients, PATHS["training_dir"])

test_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    RemapLabeld(keys=["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

test_ds = Dataset(data=test_dicts, transform=test_transforms)
test_loader = DataLoader(test_ds, batch_size=1, num_workers=0)

print(f"Test set: {len(test_ds)} patients (never used in training or validation)")

In [ ]:
# Run inference on the full test set and compute per-patient, per-class Dice

post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

class_names = {1: "NCR_NET", 2: "ED", 3: "ET"}

test_results = []

model.eval()
with torch.no_grad():
    for test_data in test_loader:
        pid = test_data["patient_id"][0]
        test_inputs = test_data["image"].to(device)
        test_labels = test_data["label"].to(device)

        test_outputs = sliding_window_inference(test_inputs, patch_size, sw_batch_size=1, predictor=model)

        pred_onehot = post_pred(test_outputs[0])
        label_onehot = post_label(test_labels[0])

        tumor_voxels = int((test_labels > 0).sum().item())

        patient_result = {"patient_id": pid, "tumor_voxels": tumor_voxels}

        overall_dice = DiceMetric(include_background=False, reduction="mean")
        overall_dice(y_pred=pred_onehot.unsqueeze(0), y=label_onehot.unsqueeze(0))
        patient_result["dice_overall"] = overall_dice.aggregate().item()

        for class_idx, class_name in class_names.items():
            class_dice = DiceMetric(include_background=True, reduction="mean")
            pred_class = pred_onehot[class_idx:class_idx+1].unsqueeze(0)
            label_class = label_onehot[class_idx:class_idx+1].unsqueeze(0)
            class_dice(y_pred=pred_class, y=label_class)
            score = class_dice.aggregate().item()
            patient_result[f"dice_{class_name}"] = score

        test_results.append(patient_result)
        print(f"{pid}: overall={patient_result['dice_overall']:.3f}, "
              f"NCR/NET={patient_result['dice_NCR_NET']:.3f}, "
              f"ED={patient_result['dice_ED']:.3f}, "
              f"ET={patient_result['dice_ET']:.3f}")

metrics_path = os.path.join(PATHS["metrics"], "final_test_dice.json")
with open(metrics_path, "w") as f:
    json.dump(test_results, f, indent=2)

print(f"\nSaved -> {metrics_path}")

In [ ]:
# Compute test set statistics and compare against validation performance

test_dice_overall = [r["dice_overall"] for r in test_results]
test_dice_ncr = [r["dice_NCR_NET"] for r in test_results]
test_dice_ed = [r["dice_ED"] for r in test_results]
test_dice_et = [r["dice_ET"] for r in test_results]

print("Test Set Dice Statistics (n=56 patients)")
print("-" * 55)
print(f"{'Class':12s} {'mean':>8s} {'median':>8s} {'min':>8s} {'max':>8s}")
print(f"{'Overall':12s} {np.mean(test_dice_overall):8.3f} {np.median(test_dice_overall):8.3f} {min(test_dice_overall):8.3f} {max(test_dice_overall):8.3f}")
print(f"{'NCR/NET':12s} {np.mean(test_dice_ncr):8.3f} {np.median(test_dice_ncr):8.3f} {min(test_dice_ncr):8.3f} {max(test_dice_ncr):8.3f}")
print(f"{'ED':12s} {np.mean(test_dice_ed):8.3f} {np.median(test_dice_ed):8.3f} {min(test_dice_ed):8.3f} {max(test_dice_ed):8.3f}")
print(f"{'ET':12s} {np.mean(test_dice_et):8.3f} {np.median(test_dice_et):8.3f} {min(test_dice_et):8.3f} {max(test_dice_et):8.3f}")

print(f"\nComparison:")
print(f"  Validation Dice (notebook 04b): 0.6863")
print(f"  Test Dice (this notebook):      {np.mean(test_dice_overall):.4f}")
print(f"  Difference: {np.mean(test_dice_overall) - 0.6863:+.4f}")

# Distribution plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(test_dice_overall, bins=20, color="steelblue", edgecolor="white")
ax.axvline(np.mean(test_dice_overall), color="coral", linestyle="--", linewidth=1.5, label=f"mean = {np.mean(test_dice_overall):.3f}")
ax.set_xlabel("Dice Score")
ax.set_ylabel("Number of patients")
ax.set_title("Test Set Dice Distribution (n=56)")
ax.legend()
plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "test_set_dice_distribution.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Build the final 3D rendering: brain context + tumor sub-regions, with a static high-quality image

from skimage import measure
from scipy import ndimage

sample_dict = [d for d in test_dicts if d["patient_id"] == "BraTS20_Training_151"][0]
sample_transformed = test_transforms(sample_dict)

input_volume = sample_transformed["image"].unsqueeze(0).to(device)
flair_volume = sample_transformed["image"][3].numpy()

with torch.no_grad():
    pred_volume = sliding_window_inference(input_volume, patch_size, sw_batch_size=1, predictor=model)
    pred_volume = torch.argmax(pred_volume, dim=1).squeeze(0).cpu().numpy()

# Keep only the largest connected tumor region (removes stray false positives)
labeled_array, num_features = ndimage.label(pred_volume > 0)
if num_features > 0:
    sizes = ndimage.sum(pred_volume > 0, labeled_array, range(1, num_features + 1))
    largest_label = np.argmax(sizes) + 1
    cleaned_mask = (labeled_array == largest_label)
else:
    cleaned_mask = pred_volume > 0

fig = go.Figure()

# Brain outline (semi-transparent context)
brain_mask = (flair_volume > np.percentile(flair_volume[flair_volume > 0], 40)).astype(float)
try:
    brain_verts, brain_faces, _, _ = measure.marching_cubes(brain_mask, level=0.5, step_size=2)
    fig.add_trace(go.Mesh3d(
        x=brain_verts[:, 0], y=brain_verts[:, 1], z=brain_verts[:, 2],
        i=brain_faces[:, 0], j=brain_faces[:, 1], k=brain_faces[:, 2],
        color="lightgray", opacity=0.10, name="Brain", showlegend=True,
    ))
except Exception as e:
    print(f"Brain outline skipped: {e}")

# Tumor sub-regions
class_colors = {1: "royalblue", 2: "gold", 3: "crimson"}
class_labels = {1: "NCR/NET (necrotic core)", 2: "ED (edema)", 3: "ET (enhancing tumor)"}

for class_id, color in class_colors.items():
    class_mask = (pred_volume == class_id) & cleaned_mask
    if class_mask.sum() < 30:
        print(f"Note: {class_labels[class_id]} not visible (too small or absent in this patient)")
        continue
    try:
        verts, faces, _, _ = measure.marching_cubes(class_mask.astype(float), level=0.5)
        fig.add_trace(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            color=color, opacity=0.9, name=class_labels[class_id], showlegend=True,
        ))
    except Exception as e:
        print(f"Skipped {class_labels[class_id]}: {e}")

axis_style = dict(backgroundcolor="rgb(15,15,15)", gridcolor="rgb(60,60,60)", showbackground=True, zeroline=False)

fig.update_layout(
    title=dict(text="3D Tumor Segmentation with Brain Context", font=dict(size=18, color="white")),
    scene=dict(
        xaxis=dict(title="X (mm)", **axis_style),
        yaxis=dict(title="Y (mm)", **axis_style),
        zaxis=dict(title="Z (mm)", **axis_style),
        bgcolor="rgb(15,15,15)",
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),
    ),
    paper_bgcolor="rgb(15,15,15)",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(0,0,0,0.6)", bordercolor="gray", borderwidth=1),
    width=850,
    height=750,
)

# Save interactive HTML (for the Streamlit app / interactive sharing)
html_path = os.path.join(PATHS["figures"], "3d_tumor_rendering.html")
fig.write_html(html_path)

# Save a static high-quality image (for the README)
png_path = os.path.join(PATHS["figures"], "3d_tumor_rendering.png")
fig.write_image(png_path, scale=2)

fig.show()
print(f"Saved interactive HTML -> {html_path}")
print(f"Saved static image -> {png_path}")

In [ ]:
# Generate a rotating GIF from the same figure (for LinkedIn / social sharing)
# Slower rotation for better visibility

import imageio.v2 as imageio

frames = []
n_frames = 60  # more frames = smoother rotation
for i in range(n_frames):
    angle = (i / n_frames) * 2 * np.pi
    camera = dict(eye=dict(x=1.8 * np.cos(angle), y=1.8 * np.sin(angle), z=0.8))
    fig.update_layout(scene_camera=camera)
    img_bytes = fig.to_image(format="png", width=700, height=600, scale=1.5)
    frames.append(imageio.imread(img_bytes))

gif_path = os.path.join(PATHS["figures"], "3d_tumor_rotation.gif")
imageio.mimsave(gif_path, frames, fps=8)  # slower fps
print(f"Saved -> {gif_path}")

In [ ]:
# Compute standard BraTS evaluation metrics: Whole Tumor, Tumor Core, Enhancing Tumor
# These are combined regions, used as the standard benchmark across BraTS literature

wt_dice_list = []
tc_dice_list = []
et_dice_list = []

model.eval()
with torch.no_grad():
    for test_data in test_loader:
        test_inputs = test_data["image"].to(device)
        test_labels = test_data["label"].to(device)

        test_outputs = sliding_window_inference(test_inputs, patch_size, sw_batch_size=1, predictor=model)
        pred_classes = torch.argmax(test_outputs, dim=1)  # shape: (1, H, W, D)

        label_raw = test_labels.squeeze(1)  # shape: (1, H, W, D), values 0,1,2,3

        # WT = union of all tumor classes (1, 2, 3)
        pred_wt = (pred_classes > 0).float()
        label_wt = (label_raw > 0).float()

        # TC = NCR/NET (1) + ET (3)
        pred_tc = ((pred_classes == 1) | (pred_classes == 3)).float()
        label_tc = ((label_raw == 1) | (label_raw == 3)).float()

        # ET = class 3 only
        pred_et = (pred_classes == 3).float()
        label_et = (label_raw == 3).float()

        def dice_score(pred, label, eps=1e-6):
            intersection = (pred * label).sum()
            return ((2. * intersection + eps) / (pred.sum() + label.sum() + eps)).item()

        wt_dice_list.append(dice_score(pred_wt, label_wt))
        tc_dice_list.append(dice_score(pred_tc, label_tc))
        et_dice_list.append(dice_score(pred_et, label_et))

print("Standard BraTS Evaluation Metrics (Test Set, n=56)")
print("-" * 55)
print(f"{'Region':20s} {'mean':>8s} {'median':>8s}")
print(f"{'WT (Whole Tumor)':20s} {np.mean(wt_dice_list):8.3f} {np.median(wt_dice_list):8.3f}")
print(f"{'TC (Tumor Core)':20s} {np.mean(tc_dice_list):8.3f} {np.median(tc_dice_list):8.3f}")
print(f"{'ET (Enhancing)':20s} {np.mean(et_dice_list):8.3f} {np.median(et_dice_list):8.3f}")
print()
print("For reference, a plain 3D U-Net baseline (Inc0mple et al., BraTS2020, 50 epochs,")
print("RTX 3090 24GB) reported: WT=0.865, TC=0.779 (approx), ET=0.641")

In [ ]:
# Final Summary

print("NOTEBOOK 05 COMPLETE — PROJECT FINAL RESULTS")
print("=" * 55)
print()
print("Model: 3D U-Net (MONAI), 4,811,129 parameters")
print("Training: 45 epochs with checkpoint/resume support")
print()
print("Results Overview (raw per-class Dice)")
print("-" * 55)
print(f"{'Metric':30s} {'Validation':>12s} {'Test':>12s}")
print(f"{'Overall Dice':30s} {'0.6863':>12s} {np.mean(test_dice_overall):>12.4f}")
print(f"{'NCR/NET Dice':30s} {'—':>12s} {np.mean(test_dice_ncr):>12.4f}")
print(f"{'ED Dice':30s} {'—':>12s} {np.mean(test_dice_ed):>12.4f}")
print(f"{'ET Dice':30s} {'—':>12s} {np.mean(test_dice_et):>12.4f}")
print()
print("Standard BraTS Evaluation Metrics (Test Set, n=56)")
print("-" * 55)
print(f"{'Region':20s} {'This model':>12s} {'Reference*':>12s}")
print(f"{'WT (Whole Tumor)':20s} {np.mean(wt_dice_list):>12.3f} {'0.865':>12s}")
print(f"{'TC (Tumor Core)':20s} {np.mean(tc_dice_list):>12.3f} {'0.779':>12s}")
print(f"{'ET (Enhancing)':20s} {np.mean(et_dice_list):>12.3f} {'0.641':>12s}")
print()
print("* Reference: plain 3D U-Net baseline, BraTS2020, 50 epochs, RTX 3090 24GB")
print("  (Inc0mple et al., github.com/Inc0mple/3D_Brain_Tumor_Seg_V2)")
print("  This model matches WT, exceeds ET, and is close on TC — using a")
print("  consumer GPU (RTX 4060, 8.6GB) with roughly 1/3 the VRAM.")
print()
print(f"Validation-to-test gap: {np.mean(test_dice_overall) - 0.6863:+.4f} (small, indicates good generalization)")
print()
print("Key findings:")
print("  - NCR/NET consistently the weakest raw class (val: notebook 04, confirmed on test)")
print("  - On standard WT/TC/ET grouping, performance is competitive with published baselines")
print("  - Tumor size not strongly correlated with performance (r=0.177)")
print("  - Extended training (45 vs 15 epochs) improved Dice by ~2.9 points before plateauing")
print()
print("Visual outputs:")
for fname in ["test_set_dice_distribution.png", "3d_tumor_rendering.png", "3d_tumor_rendering.html", "3d_tumor_rotation.gif"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Project complete. Next steps: deployment (Hugging Face + Streamlit) and README finalization.")